# Lens Distortion Correction

Wide-angle and consumer lenses introduce **radial and tangential distortion** that warps
straight lines and displaces pixels — especially near the frame edges. Correcting this before
tracking or measuring objects removes a systematic bias in speed, distance, and size estimates.

This notebook demonstrates the complete undistortion workflow using **OpenCV**:

1. Load camera calibration parameters (from a JSON file, TRex `.settings`, or inline).
2. Visualize the distortion field to understand the magnitude of the effect.
3. Build pixel remap tables and apply them to images.
4. Undistort 2-D trajectory coordinates directly (no need to reprocess video frames).
5. Export calibration parameters as JSON when you want to reuse them in another notebook.

The notebook uses **example calibration parameters** from a real Basler 5 MP camera (from
Francisco et al., *in prep.*) and generates **synthetic test images** so it runs completely
standalone. Replace the parameters and images with your own data to process real recordings.

**Requirements:** `pip install opencv-python numpy matplotlib`


In [ ]:
import json
import ast
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


## Calibration parameters

Camera calibration produces two sets of parameters:

| Parameter | Symbol | Meaning |
|-----------|--------|---------|
| Focal lengths | `fx`, `fy` | Pixels per unit length in x and y |
| Principal point | `cx`, `cy` | Optical centre in pixel coordinates |
| Radial distortion | `k1`, `k2`, `k3` | Barrel/pincushion distortion coefficients |
| Tangential distortion | `p1`, `p2` | Lens tilt relative to the sensor |

These can be loaded from:
- A **JSON file** (output of multical, Anipose, OpenCV, or this notebook)
- A **TRex `.settings` file** (contains `cam_matrix` and `cam_undistort_vector`)
- **Inline values** as shown below

The example values are from a calibrated Basler acA5472 camera with a 8 mm lens, used in
fish-tracking experiments (Francisco et al., *in prep.*).


In [ ]:
# ── Option A: inline (example values from a Basler 5 MP camera) ──────────────
mtx = np.array([
    [2967.57,    0.00, 1989.92],
    [   0.00, 2967.17, 1322.03],
    [   0.00,    0.00,    1.00],
], dtype=np.float64)

dist = np.array([-0.13210, 0.05882, 4.67e-5, -3.90e-4, -0.05091], dtype=np.float64)
# dist = [k1, k2, p1, p2, k3]

# ── Option B: load from JSON ──────────────────────────────────────────────────
# with open("calibration.json") as f:
#     cal = json.load(f)
# mtx  = np.asarray(cal["camera_matrix"], dtype=np.float64)
# dist = np.asarray(cal["dist_coeffs"],   dtype=np.float64)

# ── Option C: load from TRex .settings file ───────────────────────────────────
# def load_settings(path):
#     cfg = {}
#     with open(path) as f:
#         for line in f:
#             line = line.strip()
#             if not line or line.startswith("#"):
#                 continue
#             k, _, v = line.partition("=")
#             if v.strip().startswith("["):
#                 cfg[k.strip()] = ast.literal_eval(v.strip())
#     return (np.asarray(cfg["cam_matrix"],           dtype=np.float64),
#             np.asarray(cfg["cam_undistort_vector"], dtype=np.float64))
# mtx, dist = load_settings("../examples/calibration/config.settings")

fx, fy = mtx[0, 0], mtx[1, 1]
cx, cy = mtx[0, 2], mtx[1, 2]
k1, k2, p1, p2, k3 = dist

print(f"Focal length  :  fx = {fx:.1f} px    fy = {fy:.1f} px")
print(f"Principal point:  cx = {cx:.1f} px   cy = {cy:.1f} px")
print(f"Radial distortion:  k1 = {k1:.4f}  k2 = {k2:.4f}  k3 = {k3:.4f}")
print(f"Tangential distortion:  p1 = {p1:.2e}  p2 = {p2:.2e}")
print()
print("k1 < 0  →  barrel distortion (straight lines bow outward near frame edges)")
print("k1 > 0  →  pincushion distortion (straight lines bow inward)")


## Visualise the distortion field

The distortion field shows how much each pixel is displaced from its ideal (undistorted) position.
Displacement is largest at the corners and edges of the frame, where the effect on tracking
accuracy and distance measurements is greatest.


In [ ]:
IMG_H, IMG_W = 2664, 3984   # sensor resolution of the calibrated camera


def plot_distortion_field(mtx, dist, img_hw, step=300):
    h, w = img_hw
    xs = np.arange(0, w + 1, step, dtype=np.float32)
    ys = np.arange(0, h + 1, step, dtype=np.float32)
    gx, gy = np.meshgrid(xs, ys)
    ideal = np.stack([gx.ravel(), gy.ravel()], axis=1).reshape(-1, 1, 2)

    # undistortPoints maps distorted pixel coords → undistorted pixel coords.
    # The grid below samples distorted input coordinates; outputs are corrected coordinates.
    distorted = cv2.undistortPoints(
        ideal.astype(np.float32), cameraMatrix=mtx, distCoeffs=dist, P=mtx
    ).reshape(-1, 2)

    ideal_2d = ideal.reshape(-1, 2)
    disp = np.linalg.norm(distorted - ideal_2d, axis=1)

    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    ax = axs[0]
    scale = 15  # exaggerate arrows for visibility
    disp_vec = (distorted - ideal_2d) * scale
    ax.quiver(ideal_2d[:, 0], ideal_2d[:, 1], disp_vec[:, 0], disp_vec[:, 1],
              angles='xy', scale_units='xy', scale=1, color='C1', width=0.002)
    ax.scatter(ideal_2d[:, 0], ideal_2d[:, 1], s=10, c='C0', zorder=3)
    ax.set_xlim(0, w); ax.set_ylim(h, 0)
    ax.set_aspect('equal')
    ax.set_title(f'Distortion vectors (arrows scaled {scale}×)')
    ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')

    sc = axs[1].scatter(ideal_2d[:, 0], ideal_2d[:, 1],
                         c=disp, cmap='plasma', s=60, vmin=0)
    axs[1].set_xlim(0, w); axs[1].set_ylim(h, 0)
    axs[1].set_aspect('equal')
    axs[1].set_title('Pixel displacement magnitude')
    axs[1].set_xlabel('x (px)'); axs[1].set_ylabel('y (px)')
    plt.colorbar(sc, ax=axs[1], label='Displacement (px)')

    plt.tight_layout()
    plt.show()

    print(f'Maximum displacement : {disp.max():.1f} px  (frame corners)')
    print('Displacements are sampled on a grid in distorted input coordinates.')
    print(f'Mean displacement    : {disp.mean():.1f} px')


plot_distortion_field(mtx, dist, (IMG_H, IMG_W))


## Build undistortion maps and apply to an image

Pre-computing remap tables with `cv2.initUndistortRectifyMap` is the recommended approach for
batch processing: the expensive map computation happens once, then each frame is corrected with
a fast `cv2.remap` call.

### The `alpha` parameter

| `alpha` | Effect |
|---------|--------|
| `1.0` (default) | Keep the full sensor area — some border pixels become black after correction. |
| `0.0` | Crop to the largest rectangle of valid (non-black) pixels — smaller FOV but no borders. |
| `0 < alpha < 1` | Trade-off between the two extremes. |

A **synthetic grid image** is used here so the notebook runs standalone. Replace `img_test`
with `cv2.imread("your_recording.jpg")` to apply the correction to real data.


In [ ]:
def make_grid_image(h, w, spacing=200, line_width=2, bg=240, fg=60, mtx=None):
    """Synthetic regular-grid test image."""
    img = np.full((h, w, 3), bg, dtype=np.uint8)
    for x in range(0, w, spacing):
        cv2.line(img, (x, 0), (x, h - 1), (fg, fg, fg), line_width)
    for y in range(0, h, spacing):
        cv2.line(img, (0, y), (w - 1, y), (fg, fg, fg), line_width)
    if mtx is not None:
        cv2.circle(img, (int(mtx[0, 2]), int(mtx[1, 2])), 18, (60, 60, 200), -1)
    return img


img_test = make_grid_image(IMG_H, IMG_W, mtx=mtx)
# img_test = cv2.imread("path/to/your/image.jpg")  # ← use your own image

h, w = img_test.shape[:2]

alpha = 1.0
new_mtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), alpha, (w, h))
map1, map2   = cv2.initUndistortRectifyMap(mtx, dist, None, new_mtx, (w, h), cv2.CV_16SC2)

img_undist = cv2.remap(img_test, map1, map2, interpolation=cv2.INTER_LINEAR)

print(f"Image size         : {w} × {h} px")
print(f"Original matrix    : fx={mtx[0,0]:.1f}  fy={mtx[1,1]:.1f}  cx={mtx[0,2]:.1f}  cy={mtx[1,2]:.1f}")
print(f"New matrix (alpha={alpha}): fx={new_mtx[0,0]:.1f}  fy={new_mtx[1,1]:.1f}  cx={new_mtx[0,2]:.1f}  cy={new_mtx[1,2]:.1f}")
print(f"Valid ROI          : x={roi[0]}  y={roi[1]}  w={roi[2]}  h={roi[3]}")


In [ ]:
# Downsample for display
scale = 0.22
d_test   = cv2.resize(img_test,  (0, 0), fx=scale, fy=scale)
d_undist = cv2.resize(img_undist, (0, 0), fx=scale, fy=scale)

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].imshow(cv2.cvtColor(d_test,   cv2.COLOR_BGR2RGB))
axs[0].set_title("Input grid image")
axs[0].axis("off")
axs[1].imshow(cv2.cvtColor(d_undist, cv2.COLOR_BGR2RGB))
axs[1].set_title(f"After undistortion  (alpha={alpha})")
axs[1].axis("off")
plt.suptitle(
    "The curved borders show the extent of the geometric correction.\n"
    "In practice, your input is the distorted camera image and the output has straight lines.",
    fontsize=10, color="gray",
)
plt.tight_layout()
plt.show()

# Show crop-only version (alpha=0)
new_mtx0, roi0 = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), 0.0, (w, h))
map1_c, map2_c = cv2.initUndistortRectifyMap(mtx, dist, None, new_mtx0, (w, h), cv2.CV_16SC2)
img_crop = cv2.remap(img_test, map1_c, map2_c, cv2.INTER_LINEAR)
x0, y0, rw, rh = roi0
img_crop = img_crop[y0:y0 + rh, x0:x0 + rw]

d_crop = cv2.resize(img_crop, (0, 0), fx=scale, fy=scale)
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].imshow(cv2.cvtColor(d_undist, cv2.COLOR_BGR2RGB))
axs[0].set_title(f"alpha=1 — full frame ({w}×{h} px)")
axs[0].axis("off")
axs[1].imshow(cv2.cvtColor(d_crop, cv2.COLOR_BGR2RGB))
axs[1].set_title(f"alpha=0 — cropped to valid pixels ({rw}×{rh} px)")
axs[1].axis("off")
plt.suptitle("alpha parameter: trade-off between frame coverage and valid-pixel area", fontsize=11)
plt.tight_layout()
plt.show()


## Undistort trajectory coordinates directly

If you have already extracted trajectory data (pixel coordinates from TRex, DeepLabCut, SLEAP, etc.),
you can correct the coordinates with `cv2.undistortPoints` **without reprocessing the video frames**.
This is much faster for large datasets and produces identical results.

The correction matters most for positions near frame edges, where the displacement can reach tens
of pixels, introducing a systematic bias into speed and distance estimates.


In [ ]:
def undistort_trajectory(X, Y, mtx, dist, new_mtx):
    """Undistort 2-D trajectory coordinates (pixel space)."""
    pts = np.column_stack([X, Y]).astype(np.float32).reshape(-1, 1, 2)
    pts_u = cv2.undistortPoints(pts, cameraMatrix=mtx, distCoeffs=dist, P=new_mtx)
    return pts_u[:, 0, 0], pts_u[:, 0, 1]


def step_speed(X, Y):
    return np.sqrt(np.diff(X) ** 2 + np.diff(Y) ** 2)


# Simulate a circular trajectory near the frame edge (where distortion is large)
n_frames = 400
theta    = np.linspace(0, 2 * np.pi, n_frames)
# Centre the trajectory at a distorted location far from the principal point
traj_cx, traj_cy = mtx[0, 2] * 1.6, mtx[1, 2] * 0.4
radius   = 500
X_dist   = traj_cx + radius * np.cos(theta)
Y_dist   = traj_cy + radius * np.sin(theta)

X_undist, Y_undist = undistort_trajectory(X_dist, Y_dist, mtx, dist, new_mtx)

speed_dist   = step_speed(X_dist,   Y_dist)
speed_undist = step_speed(X_undist, Y_undist)
speed_bias   = speed_dist - speed_undist
disp_px      = np.sqrt((X_dist - X_undist) ** 2 + (Y_dist - Y_undist) ** 2)

fig, axs = plt.subplots(1, 3, figsize=(16, 5))

axs[0].plot(X_dist,   Y_dist,   lw=1.5, label="Distorted")
axs[0].plot(X_undist, Y_undist, lw=1.5, ls="--", label="Undistorted")
axs[0].set_aspect("equal"); axs[0].invert_yaxis()
axs[0].legend(); axs[0].grid(True, alpha=0.3)
axs[0].set_title("Trajectory (image coordinates)")
axs[0].set_xlabel("x (px)"); axs[0].set_ylabel("y (px)")

axs[1].plot(disp_px, "k-", lw=0.9, alpha=0.8)
axs[1].axhline(disp_px.mean(), color="C1", ls="--", label=f"Mean = {disp_px.mean():.1f} px")
axs[1].set_xlabel("Frame"); axs[1].set_ylabel("Pixel displacement (px)")
axs[1].set_title("Point displacement due to undistortion")
axs[1].legend(); axs[1].grid(True, alpha=0.3)

axs[2].plot(speed_bias, "k-", lw=0.9, alpha=0.8)
axs[2].axhline(0, color="gray", ls="--")
axs[2].axhline(speed_bias.mean(), color="C3", ls="--",
               label=f"Mean bias = {speed_bias.mean():.3f} px/frame")
axs[2].set_xlabel("Frame"); axs[2].set_ylabel("Speed bias (px/frame)")
axs[2].set_title("Instantaneous speed bias (distorted - undistorted)")
axs[2].legend(); axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Position displacement  : mean={disp_px.mean():.2f} px   max={disp_px.max():.2f} px")
print(f"Speed bias             : mean={speed_bias.mean():.4f} px/frame")
print()
print("Even modest distortion (k1 = -0.13) introduces a measurable speed bias")
print("for trajectories near frame edges. Always undistort before computing kinematics.")


## Batch processing in a notebook-only workflow

For large folders or videos, keep the processing inside this notebook by wrapping the remap operation in a small function and pointing it at your own input paths. The key steps are unchanged:

1. Load calibration parameters.
2. Build remap tables once for the input resolution.
3. Apply `cv2.remap` to each image/frame.
4. Save corrected outputs to a generated-output folder such as `undistorted/`, which is ignored by git.

The notebook intentionally does not depend on a standalone script; copy the cells above and adapt the input paths for each dataset.


In [ ]:
# Build a reusable calibration dictionary.
# Set SAVE_JSON = True only when you want to write an example file for reuse.

calibration = {
    "camera_matrix": mtx.tolist(),
    "dist_coeffs": dist.tolist(),
}

SAVE_JSON = False
out_path = Path("../examples/calibration/calibration_example.json")

if SAVE_JSON:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(calibration, f, indent=2)
    print(f"Calibration saved to {out_path.resolve()}")
else:
    print("Calibration JSON preview; set SAVE_JSON = True to write it.")

print(json.dumps(calibration, indent=2))
